# 📊 Veri Kayması (Data Drift) Tespiti — A'dan Z'ye Rehber

---

## 🎯 Bu Notebook'ta Ne Öğreneceksiniz?

| Bölüm | Konu |
|-------|------|
| 1 | Data Drift nedir? Neden önemlidir? |
| 2 | Drift türleri (Covariate, Label, Concept) |
| 3 | Sentetik veri üretimi (referans vs. production) |
| 4 | İstatistiksel testler (KS, Chi-Square, PSI, JS Divergence) |
| 5 | Görselleştirme teknikleri |
| 6 | Evidently AI ile otomatik drift raporu |
| 7 | Makine öğrenmesi tabanlı drift tespiti |
| 8 | Gerçek dünya senaryosu (End-to-End) |
| 9 | Drift'e karşı önlemler ve izleme stratejileri |

---

> **📌 Tanım:** *Data Drift (Veri Kayması)*, bir ML modelinin üretimde (production) gördüğü verinin,
> modelin eğitildiği veriden istatistiksel olarak farklılaşması durumudur.
> Bu durum model performansının bozulmasına neden olur.

## 📦 Bölüm 0: Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Gerekli kütüphaneleri kur
# Not: Zaten kuruluysa bu hücreyi atlayabilirsiniz
!pip install evidently scikit-learn scipy numpy pandas matplotlib seaborn --quiet

In [ ]:
# =============================================================================
# TEMEL IMPORT'LAR
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency
from scipy.spatial.distance import jensenshannon

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Tekrar üretilebilirlik için seed sabitle
np.random.seed(42)

# Grafik ayarları
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
sns.set_style('whitegrid')

print("✅ Tüm kütüphaneler başarıyla yüklendi!")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

---
## 📚 Bölüm 1: Data Drift Nedir?

### 🔍 Temel Kavramlar

```
EĞİTİM VERİSİ (Training Data)
        ↓
    ML Modeli Eğitilir
        ↓
ÜRETİM VERİSİ (Production Data)  ←── ZAMAN İÇİNDE DEĞİŞİR!
        ↓
  Model Performansı DÜŞER  ← DATA DRIFT!
```

### 🏷️ Drift Türleri

| Tür | Açıklama | Formül |
|-----|----------|--------|
| **Covariate Drift** | Giriş değişkenlerinin (X) dağılımı değişir | P_train(X) ≠ P_prod(X) |
| **Label Drift** | Hedef değişkenin (y) dağılımı değişir | P_train(y) ≠ P_prod(y) |
| **Concept Drift** | X ile y arasındaki ilişki değişir | P_train(y|X) ≠ P_prod(y|X) |
| **Feature Drift** | Belirli bir özelliğin dağılımı değişir | P_train(x_i) ≠ P_prod(x_i) |

In [ ]:
# =============================================================================
# BÖLÜM 1: Drift Türlerini Görsel Olarak Anlayalım
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(-5, 10, 300)

# --- 1. Covariate Drift ---
ax = axes[0]
ax.plot(x, stats.norm.pdf(x, loc=0, scale=1), 'b-', lw=2.5, label='Eğitim (μ=0)')
ax.plot(x, stats.norm.pdf(x, loc=3, scale=1.5), 'r--', lw=2.5, label='Üretim (μ=3)')
ax.fill_between(x, stats.norm.pdf(x, loc=0, scale=1), alpha=0.2, color='blue')
ax.fill_between(x, stats.norm.pdf(x, loc=3, scale=1.5), alpha=0.2, color='red')
ax.set_title('1️⃣ Covariate Drift\n(X dağılımı kayıyor)', fontweight='bold')
ax.set_xlabel('Özellik Değeri (X)')
ax.set_ylabel('Yoğunluk')
ax.legend()

# --- 2. Label Drift ---
ax = axes[1]
categories = ['Sınıf 0', 'Sınıf 1', 'Sınıf 2']
train_probs = [0.6, 0.3, 0.1]
prod_probs  = [0.2, 0.2, 0.6]
x_pos = np.arange(len(categories))
bars1 = ax.bar(x_pos - 0.2, train_probs, 0.35, label='Eğitim', color='steelblue', alpha=0.8)
bars2 = ax.bar(x_pos + 0.2, prod_probs,  0.35, label='Üretim', color='tomato', alpha=0.8)
ax.set_title('2️⃣ Label Drift\n(y dağılımı kayıyor)', fontweight='bold')
ax.set_xlabel('Sınıf')
ax.set_ylabel('Olasılık')
ax.set_xticks(x_pos)
ax.set_xticklabels(categories)
ax.legend()

# --- 3. Concept Drift ---
ax = axes[2]
x_vals = np.linspace(0, 10, 100)
y_train = 2 * x_vals + 1 + np.random.normal(0, 0.5, 100)   # Pozitif ilişki
y_prod  = -1.5 * x_vals + 15 + np.random.normal(0, 0.5, 100)  # Negatif ilişki!
ax.scatter(x_vals[::5], y_train[::5], color='blue', alpha=0.6, s=40, label='Eğitim: y=2x+1')
ax.scatter(x_vals[::5], y_prod[::5],  color='red',  alpha=0.6, s=40, marker='^', label='Üretim: y=-1.5x+15')
ax.set_title('3️⃣ Concept Drift\n(X→y ilişkisi değişiyor)', fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('y')
ax.legend()

plt.tight_layout()
plt.suptitle('Data Drift Türleri', fontsize=15, fontweight='bold', y=1.02)
plt.show()
print("📌 Her üç türde de eğitim ve üretim dağılımları birbirinden ayrışmaktadır.")

---
## 🏗️ Bölüm 2: Sentetik Veri Üretimi

Gerçekçi bir senaryo oluşturacağız:
- **Senaryo:** Bir bankanın kredi riski modeli
- **Referans veri:** Modelin eğitildiği dönem (Ocak-Haziran)
- **Production veri:** Modelin çalıştığı dönem (Temmuz-Aralık) — **ekonomik kriz yaşanıyor!**

In [ ]:
# =============================================================================
# BÖLÜM 2: Senaryo — Banka Kredi Riski Modeli
# Referans: Normal dönem | Production: Ekonomik kriz dönemi
# =============================================================================

def uret_kredi_verisi(n=2000, kriz_modu=False, seed=42):
    """
    Banka kredi başvurusu verisi üretir.
    
    Parametreler:
    -----------
    n         : Üretilecek kayıt sayısı
    kriz_modu : True ise dağılımlar ekonomik krizi yansıtacak şekilde değişir
    seed      : Rastgelelik için sabit değer
    """
    np.random.seed(seed)
    
    if not kriz_modu:
        # ---- NORMAL DÖNEM (Referans / Training Data) ----
        gelir       = np.random.normal(loc=55000, scale=15000, size=n).clip(10000, 200000)
        kredi_skoru = np.random.normal(loc=680,   scale=80,    size=n).clip(300, 850).astype(int)
        istihdam_yil= np.random.exponential(scale=5, size=n).clip(0, 40)
        borclanma   = np.random.beta(a=2, b=5, size=n)  # Çoğunluk düşük borçlu
        yasinda_ev  = np.random.choice([0, 1], size=n, p=[0.35, 0.65])  # %65 ev sahibi
        egitim      = np.random.choice(['Lise', 'Lisans', 'Yükseklisans'],
                                       size=n, p=[0.30, 0.50, 0.20])
        temerrut_prob = 1 / (1 + np.exp(
            0.00003 * gelir - 0.01 * kredi_skoru + 0.5 * borclanma - 0.5
        ))
        temerrut = (np.random.uniform(size=n) < temerrut_prob).astype(int)
        
    else:
        # ---- KRİZ DÖNEMİ (Production Data — DRİFT VAR!) ----
        gelir       = np.random.normal(loc=42000, scale=20000, size=n).clip(5000, 200000)  # Gelirler düştü!
        kredi_skoru = np.random.normal(loc=620,   scale=100,   size=n).clip(300, 850).astype(int)  # Skorlar geriledi!
        istihdam_yil= np.random.exponential(scale=3, size=n).clip(0, 40)  # İşsizlik arttı!
        borclanma   = np.random.beta(a=4, b=3, size=n)  # Borçlanma oranı yükseldi!
        yasinda_ev  = np.random.choice([0, 1], size=n, p=[0.55, 0.45])  # Ev sahipliği düştü!
        egitim      = np.random.choice(['Lise', 'Lisans', 'Yükseklisans'],
                                       size=n, p=[0.45, 0.45, 0.10])  # Dağılım değişti!
        temerrut_prob = 1 / (1 + np.exp(
            0.00003 * gelir - 0.008 * kredi_skoru + 0.8 * borclanma - 0.2  # İlişki de değişti!
        ))
        temerrut = (np.random.uniform(size=n) < temerrut_prob).astype(int)
    
    df = pd.DataFrame({
        'gelir':           gelir.round(0),
        'kredi_skoru':     kredi_skoru,
        'istihdam_yil':    istihdam_yil.round(1),
        'borclanma_orani': borclanma.round(4),
        'ev_sahibi':       yasinda_ev,
        'egitim_duzeyi':   egitim,
        'temerrut':        temerrut
    })
    return df

# Verileri üret
referans_df   = uret_kredi_verisi(n=2000, kriz_modu=False)  # Eğitim dönemi
production_df = uret_kredi_verisi(n=2000, kriz_modu=True)   # Kriz dönemi

print("📊 VERİ SETİ BİLGİLERİ")
print("="*50)
print(f"Referans (Eğitim) veri:\t {referans_df.shape[0]:,} kayıt, {referans_df.shape[1]} sütun")
print(f"Production (Üretim) veri:\t {production_df.shape[0]:,} kayıt, {production_df.shape[1]} sütun")
print()
print("Referans veri — İlk 5 satır:")
referans_df.head()

In [ ]:
# =============================================================================
# Temel istatistiksel karşılaştırma
# =============================================================================

sayisal_sutunlar = ['gelir', 'kredi_skoru', 'istihdam_yil', 'borclanma_orani']

# İki veri setinin özet istatistiklerini yan yana göster
ref_stats  = referans_df[sayisal_sutunlar].describe().round(2)
prod_stats = production_df[sayisal_sutunlar].describe().round(2)

karsilastirma = pd.concat([ref_stats, prod_stats], keys=['🔵 Referans', '🔴 Production'], axis=1)
print("📈 TEMEL İSTATİSTİKSEL KARŞILAŞTIRMA")
print("="*60)
karsilastirma

---
## 📐 Bölüm 3: İstatistiksel Drift Tespit Yöntemleri

### 3.1 Kolmogorov-Smirnov (KS) Testi

> **KS Testi**, iki sürekli dağılımın istatistiksel olarak aynı olup olmadığını test eder.
> İki CDF (kümülatif dağılım fonksiyonu) arasındaki maksimum farkı ölçer.

$$D_{KS} = \sup_x |F_{ref}(x) - F_{prod}(x)|$$

- **p-value < 0.05** → Drift var (dağılımlar istatistiksel olarak farklı)
- **p-value ≥ 0.05** → Drift yok (dağılımlar benzer)

In [ ]:
# =============================================================================
# BÖLÜM 3.1: Kolmogorov-Smirnov (KS) Testi
# Sürekli değişkenler için en yaygın kullanılan drift tespit yöntemi
# =============================================================================

def ks_testi_uygula(ref_df, prod_df, sutunlar):
    """
    Belirtilen sütunlar için KS testi uygular ve sonuçları DataFrame olarak döndürür.
    
    KS istatistiği: 0'a yakın = benzer dağılım, 1'e yakın = çok farklı dağılım
    """
    sonuclar = []
    
    for sutun in sutunlar:
        ks_stat, p_value = ks_2samp(
            ref_df[sutun].dropna(), 
            prod_df[sutun].dropna()
        )
        
        # Drift şiddetini sınıflandır
        if ks_stat < 0.1:
            siddet = '🟢 Düşük'
        elif ks_stat < 0.2:
            siddet = '🟡 Orta'
        else:
            siddet = '🔴 Yüksek'
        
        sonuclar.append({
            'Özellik':          sutun,
            'KS İstatistiği':   round(ks_stat, 4),
            'p-değeri':         round(p_value, 6),
            'Drift Var mı?':    'EVET ⚠️' if p_value < 0.05 else 'HAYIR ✅',
            'Drift Şiddeti':    siddet
        })
    
    return pd.DataFrame(sonuclar)

# Sayısal sütunlar için KS testi
ks_sonuclari = ks_testi_uygula(referans_df, production_df, sayisal_sutunlar)

print("🔬 KOLMOGOROV-SMIRNOV (KS) TEST SONUÇLARI")
print("="*65)
print(ks_sonuclari.to_string(index=False))

In [ ]:
# =============================================================================
# KS Testini Görselleştir — CDF Karşılaştırması
# CDF = Cumulative Distribution Function (Kümülatif Dağılım Fonksiyonu)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, sutun in enumerate(sayisal_sutunlar):
    ax = axes[idx]
    
    ref_vals  = np.sort(referans_df[sutun].dropna())
    prod_vals = np.sort(production_df[sutun].dropna())
    
    # CDF eğrileri
    ref_cdf  = np.arange(1, len(ref_vals)  + 1) / len(ref_vals)
    prod_cdf = np.arange(1, len(prod_vals) + 1) / len(prod_vals)
    
    ax.plot(ref_vals,  ref_cdf,  'b-',  lw=2, label='🔵 Referans (Eğitim)')
    ax.plot(prod_vals, prod_cdf, 'r--', lw=2, label='🔴 Production (Kriz)')
    
    # KS istatistiğini göster
    ks_stat = ks_sonuclari.loc[ks_sonuclari['Özellik'] == sutun, 'KS İstatistiği'].values[0]
    p_val   = ks_sonuclari.loc[ks_sonuclari['Özellik'] == sutun, 'p-değeri'].values[0]
    renk    = 'red' if p_val < 0.05 else 'green'
    
    ax.set_title(f'{sutun.upper()}\nKS={ks_stat:.4f} | p={p_val:.4f}', 
                 fontweight='bold', color=renk)
    ax.set_xlabel('Değer')
    ax.set_ylabel('Kümülatif Olasılık')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Drift varsa arka planı kırmızıya boya
    if p_val < 0.05:
        ax.set_facecolor('#fff5f5')

plt.suptitle('KS Testi — CDF Karşılaştırmaları\n(Eğriler ne kadar ayrışırsa Drift o kadar fazla)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Ki-Kare (Chi-Square) Testi

> **Ki-Kare Testi**, kategorik değişkenlerin dağılımının değişip değişmediğini test eder.

$$\chi^2 = \sum_{i} \frac{(O_i - E_i)^2}{E_i}$$

- $O_i$ = Gözlenen frekans (Production)
- $E_i$ = Beklenen frekans (Referans)
- **p-value < 0.05** → Kategorik dağılım değişmiş, drift var!

In [ ]:
# =============================================================================
# BÖLÜM 3.2: Ki-Kare (Chi-Square) Testi — Kategorik Değişkenler İçin
# =============================================================================

def chi_square_testi(ref_df, prod_df, kategorik_sutunlar):
    """
    Kategorik değişkenler için Ki-Kare drift testi uygular.
    İki dağılımın frekans tablosunu karşılaştırır.
    """
    sonuclar = []
    
    for sutun in kategorik_sutunlar:
        # Her kategorinin frekansını hesapla
        ref_counts  = ref_df[sutun].value_counts().sort_index()
        prod_counts = prod_df[sutun].value_counts().sort_index()
        
        # Tüm kategorileri birleştir (bazı kategoriler bir veri setinde olmayabilir)
        tum_kategoriler = ref_counts.index.union(prod_counts.index)
        ref_counts  = ref_counts.reindex(tum_kategoriler, fill_value=0)
        prod_counts = prod_counts.reindex(tum_kategoriler, fill_value=0)
        
        # Ki-Kare testi
        contingency_table = np.array([ref_counts.values, prod_counts.values])
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        sonuclar.append({
            'Özellik':        sutun,
            'Chi² İstatistiği': round(chi2, 4),
            'p-değeri':       round(p_value, 6),
            'Serbestlik Derecesi': dof,
            'Drift Var mı?':  'EVET ⚠️' if p_value < 0.05 else 'HAYIR ✅'
        })
    
    return pd.DataFrame(sonuclar)

# Kategorik ve ikili sütunlar
kategorik_sutunlar = ['egitim_duzeyi', 'ev_sahibi', 'temerrut']
chi_sonuclari = chi_square_testi(referans_df, production_df, kategorik_sutunlar)

print("🔬 KI-KARE (CHI-SQUARE) TEST SONUÇLARI")
print("="*65)
print(chi_sonuclari.to_string(index=False))
print()
print("💡 Not: 'temerrut' sütunundaki drift = Label Drift (Hedef değişken kayması)")

In [ ]:
# =============================================================================
# Kategorik Drift Görselleştirme
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, sutun in enumerate(kategorik_sutunlar):
    ax = axes[idx]
    
    # Yüzde dağılımları hesapla
    ref_pct  = (referans_df[sutun].value_counts(normalize=True) * 100).sort_index()
    prod_pct = (production_df[sutun].value_counts(normalize=True) * 100).sort_index()
    
    tum_cat = sorted(set(ref_pct.index) | set(prod_pct.index))
    ref_pct  = ref_pct.reindex(tum_cat, fill_value=0)
    prod_pct = prod_pct.reindex(tum_cat, fill_value=0)
    
    x = np.arange(len(tum_cat))
    genislik = 0.35
    
    bars1 = ax.bar(x - genislik/2, ref_pct.values,  genislik, label='🔵 Referans', 
                   color='steelblue', alpha=0.85)
    bars2 = ax.bar(x + genislik/2, prod_pct.values, genislik, label='🔴 Production', 
                   color='tomato', alpha=0.85)
    
    # Bar üstüne değerleri yaz
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8, color='steelblue')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8, color='tomato')
    
    p_val = chi_sonuclari.loc[chi_sonuclari['Özellik'] == sutun, 'p-değeri'].values[0]
    baslik_renk = 'red' if p_val < 0.05 else 'green'
    drift_etiketi = '⚠️ DRIFT VAR' if p_val < 0.05 else '✅ DRIFT YOK'
    
    ax.set_title(f'{sutun.upper()}\n{drift_etiketi} (p={p_val:.4f})', 
                 fontweight='bold', color=baslik_renk)
    ax.set_xlabel('Kategori')
    ax.set_ylabel('Yüzde (%)')
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in tum_cat], rotation=15)
    ax.legend(fontsize=9)
    
    if p_val < 0.05:
        ax.set_facecolor('#fff5f5')

plt.suptitle('Ki-Kare Testi — Kategorik Değişken Drift Karşılaştırması', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Population Stability Index (PSI)

> **PSI (Nüfus Kararlılık İndeksi)**, finans sektöründe yaygın kullanılan bir drift ölçüsüdür.
> Özellikle kredi skorlaması ve risk modellerinde standart izleme metriğidir.

$$PSI = \sum_{i=1}^{N} \left( \%Üretim_i - \%Referans_i \right) \cdot \ln\left( \frac{\%Üretim_i}{\%Referans_i} \right)$$

| PSI Değeri | Yorum |
|-----------|-------|
| PSI < 0.1 | 🟢 Değişim yok / önemsiz |
| 0.1 ≤ PSI < 0.2 | 🟡 Orta düzey değişim — inceleme gerekli |
| PSI ≥ 0.2 | 🔴 Büyük değişim — model yeniden eğitilmeli! |

In [ ]:
# =============================================================================
# BÖLÜM 3.3: Population Stability Index (PSI)
# Finans sektörünün standart model izleme metriği
# =============================================================================

def psi_hesapla(referans, uretim, n_bins=10):
    """
    İki sayısal dağılım arasındaki PSI değerini hesaplar.
    
    Parametreler:
    -----------
    referans : Referans (eğitim) veri dizisi
    uretim   : Production (üretim) veri dizisi  
    n_bins   : Histogram bin sayısı (varsayılan: 10)
    
    Döndürür:
    --------
    psi_toplam : Toplam PSI değeri
    psi_detay  : Her bin için PSI değerleri içeren DataFrame
    """
    # Referans veriye göre bin sınırlarını belirle
    _, bin_sinirlar = np.histogram(referans, bins=n_bins)
    bin_sinirlar[0]  = -np.inf  # İlk sınırı -sonsuz yap
    bin_sinirlar[-1] =  np.inf  # Son sınırı +sonsuz yap
    
    # Her bin için frekans hesapla
    ref_counts,  _ = np.histogram(referans, bins=bin_sinirlar)
    prod_counts, _ = np.histogram(uretim,   bins=bin_sinirlar)
    
    # Sıfır bölme hatasını önlemek için küçük bir epsilon ekle
    epsilon = 1e-10
    ref_pct  = ref_counts  / len(referans) + epsilon
    prod_pct = prod_counts / len(uretim)   + epsilon
    
    # PSI formülü: Σ (üretim% - referans%) × ln(üretim% / referans%)
    psi_bins = (prod_pct - ref_pct) * np.log(prod_pct / ref_pct)
    psi_toplam = np.sum(psi_bins)
    
    # Detay tablosu
    detay = pd.DataFrame({
        'Bin': [f'Bin {i+1}' for i in range(n_bins)],
        'Referans (%)':  (ref_pct  * 100).round(2),
        'Üretim (%)':    (prod_pct * 100).round(2),
        'PSI (Bin)':     psi_bins.round(5)
    })
    
    return psi_toplam, detay

def psi_yorumla(psi):
    """PSI değerini yorumlar ve öneri üretir."""
    if psi < 0.1:
        return '🟢 Düşük (Drift yok)', 'Model güvenle kullanılabilir'
    elif psi < 0.2:
        return '🟡 Orta (Dikkat)', 'Veri değişimi izlenmeli, model gözden geçirilmeli'
    else:
        return '🔴 Yüksek (Kritik)', 'Model yeniden eğitilmeli!'

# Tüm sayısal sütunlar için PSI hesapla
print("📊 PSI (POPULATION STABILITY INDEX) SONUÇLARI")
print("="*65)

psi_ozet = []
for sutun in sayisal_sutunlar:
    psi_val, _ = psi_hesapla(
        referans_df[sutun].values, 
        production_df[sutun].values
    )
    durum, oneri = psi_yorumla(psi_val)
    psi_ozet.append({
        'Özellik': sutun,
        'PSI Değeri': round(psi_val, 4),
        'Durum': durum,
        'Öneri': oneri
    })
    print(f"  {sutun:20s}: PSI = {psi_val:.4f}  →  {durum}")

psi_df = pd.DataFrame(psi_ozet)

In [ ]:
# =============================================================================
# PSI Görselleştirme — Yatay Bar Chart
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sol: PSI Bar Chart
ax = axes[0]
renkler = ['#2ecc71' if v < 0.1 else '#f39c12' if v < 0.2 else '#e74c3c' 
           for v in psi_df['PSI Değeri']]

bars = ax.barh(psi_df['Özellik'], psi_df['PSI Değeri'], color=renkler, alpha=0.85, height=0.5)

# Eşik çizgileri
ax.axvline(x=0.1, color='orange', linestyle='--', lw=2, label='Orta eşik (0.1)')
ax.axvline(x=0.2, color='red',    linestyle='--', lw=2, label='Kritik eşik (0.2)')

# Bar üstüne değerleri yaz
for bar, val in zip(bars, psi_df['PSI Değeri']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('PSI Değeri')
ax.set_title('PSI Sonuçları\n(Özellik Başına)', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, max(psi_df['PSI Değeri']) * 1.3)

# Arka plan rengi ile bölgeler
ax.axvspan(0,   0.1, alpha=0.05, color='green')
ax.axvspan(0.1, 0.2, alpha=0.05, color='orange')
ax.axvspan(0.2, 10,  alpha=0.05, color='red')

# Sağ: 'gelir' sütunu için detaylı bin analizi
ax = axes[1]
_, gelir_psi_detay = psi_hesapla(referans_df['gelir'].values, production_df['gelir'].values)

x_pos = np.arange(len(gelir_psi_detay))
bar1 = ax.bar(x_pos - 0.2, gelir_psi_detay['Referans (%)'], 0.35, 
              label='Referans (%)', color='steelblue', alpha=0.8)
bar2 = ax.bar(x_pos + 0.2, gelir_psi_detay['Üretim (%)'],  0.35, 
              label='Üretim (%)',   color='tomato', alpha=0.8)

ax.set_title('GELİR — Bin Bazında Dağılım Karşılaştırması\n(PSI hesaplama detayı)', fontweight='bold')
ax.set_xlabel('Bin')
ax.set_ylabel('Yüzde (%)')
ax.set_xticks(x_pos)
ax.set_xticklabels(gelir_psi_detay['Bin'], rotation=45, fontsize=8)
ax.legend()

plt.suptitle('PSI Analizi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Jensen-Shannon (JS) Divergence

> **JS Divergence**, iki olasılık dağılımı arasındaki simetrik uzaklığı ölçer.
> KL Divergence'ın simetrik versiyonu olup her zaman 0-1 arasında değer alır.

$$JSD(P \| Q) = \frac{1}{2} D_{KL}(P \| M) + \frac{1}{2} D_{KL}(Q \| M), \quad M = \frac{P+Q}{2}$$

- **JSD = 0** → Dağılımlar özdeş
- **JSD = 1** → Dağılımlar tamamen farklı (kesişim yok)

In [ ]:
# =============================================================================
# BÖLÜM 3.4: Jensen-Shannon (JS) Divergence
# Hem sürekli hem kategorik değişkenler için kullanılabilir
# =============================================================================

def js_divergence_hesapla(referans, uretim, n_bins=50):
    """
    İki dağılım arasındaki Jensen-Shannon Divergence değerini hesaplar.
    
    Sayısal değişkenler için histograma dönüştürülüp olasılık vektörü oluşturulur.
    JSD = sqrt(JS) ile 0-1 arasına normalize edilmiş mesafe elde edilir.
    """
    # Her iki veri setinin ortak sınırlarını bul
    tum_degerler = np.concatenate([referans, uretim])
    min_val, max_val = tum_degerler.min(), tum_degerler.max()
    
    bin_sinirlar = np.linspace(min_val, max_val, n_bins + 1)
    
    # Her iki dağılım için aynı bin sınırlarıyla histogram
    ref_hist,  _ = np.histogram(referans, bins=bin_sinirlar, density=True)
    prod_hist, _ = np.histogram(uretim,   bins=bin_sinirlar, density=True)
    
    # Sıfır olasılıkları küçük değerle doldur (log hesabı için)
    epsilon   = 1e-10
    ref_prob  = ref_hist  + epsilon
    prod_prob = prod_hist + epsilon
    
    # Normalize et
    ref_prob  /= ref_prob.sum()
    prod_prob /= prod_prob.sum()
    
    # JS Distance (karekök alarak 0-1 arasına çek)
    jsd = jensenshannon(ref_prob, prod_prob)
    return jsd

# Tüm sayısal değişkenler için JS Divergence
print("📊 JENSEN-SHANNON DIVERGENCE SONUÇLARI")
print("="*55)
print(f"{'Özellik':20s} {'JSD':>8s}  Yorum")
print("-"*55)

jsd_sonuclari = []
for sutun in sayisal_sutunlar:
    jsd = js_divergence_hesapla(
        referans_df[sutun].values,
        production_df[sutun].values
    )
    
    if jsd < 0.1:
        yorum = '🟢 Benzer dağılım'
    elif jsd < 0.2:
        yorum = '🟡 Orta drift'
    else:
        yorum = '🔴 Yüksek drift'
    
    jsd_sonuclari.append({'Özellik': sutun, 'JSD': round(jsd, 4), 'Yorum': yorum})
    print(f"  {sutun:20s} {jsd:>8.4f}  {yorum}")

print()
print("💡 JSD = 0 → Özdeş dağılımlar | JSD = 1 → Tamamen farklı")

---
## 📊 Bölüm 4: Kapsamlı Görselleştirme

In [ ]:
# =============================================================================
# BÖLÜM 4: Histogramlarla Dağılım Karşılaştırması
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, sutun in enumerate(sayisal_sutunlar):
    ax = axes[idx]
    
    # KDE (Kernel Density Estimation) grafiği
    referans_df[sutun].plot(kind='density', ax=ax, color='steelblue', lw=2.5,
                             label='🔵 Referans (Eğitim)', alpha=0.8)
    production_df[sutun].plot(kind='density', ax=ax, color='tomato', lw=2.5,
                              label='🔴 Production (Kriz)', linestyle='--', alpha=0.8)
    
    # Ortalamalar
    ref_mean  = referans_df[sutun].mean()
    prod_mean = production_df[sutun].mean()
    ymax = ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 1
    
    ax.axvline(ref_mean,  color='steelblue', linestyle=':', lw=1.5, alpha=0.7,
               label=f'Ort. Ref: {ref_mean:.1f}')
    ax.axvline(prod_mean, color='tomato',    linestyle=':', lw=1.5, alpha=0.7,
               label=f'Ort. Prod: {prod_mean:.1f}')
    
    # Başlıkta tüm drift metriklerini göster
    ks_val  = ks_sonuclari.loc[ks_sonuclari['Özellik']  == sutun, 'KS İstatistiği'].values[0]
    psi_val = psi_df.loc[psi_df['Özellik'] == sutun, 'PSI Değeri'].values[0]
    jsd_val = next(r['JSD'] for r in jsd_sonuclari if r['Özellik'] == sutun)
    
    renk = 'red' if ks_val > 0.1 else 'green'
    ax.set_title(f'{sutun.upper()}\nKS={ks_val:.3f} | PSI={psi_val:.3f} | JSD={jsd_val:.3f}',
                 fontweight='bold', color=renk, fontsize=10)
    ax.set_ylabel('Yoğunluk')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Dağılım Karşılaştırması (KDE Grafikleri)\nEğitim vs. Production — Ekonomik Kriz Senaryosu',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Drift Özet Dashboard — Isı Haritası
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Normalize edilmiş drift skorları ısı haritası
ax = axes[0]

# Her metrik için normalize et (0-1 arası)
ks_vals  = ks_sonuclari.set_index('Özellik')['KS İstatistiği']
psi_vals = psi_df.set_index('Özellik')['PSI Değeri']
jsd_vals = pd.Series({r['Özellik']: r['JSD'] for r in jsd_sonuclari})

# PSI'yı 0-1'e normalize et (PSI max ~2 kabul edelim)
heatmap_data = pd.DataFrame({
    'KS (0-1)':      ks_vals,
    'PSI/2 (0-1)':   (psi_vals / 2).clip(0, 1),
    'JSD (0-1)':     jsd_vals
})

sns.heatmap(heatmap_data.T, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.5, ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Drift Şiddeti (Kırmızı = Yüksek)'})
ax.set_title('Drift Şiddeti Isı Haritası\n(Her Metrik İçin Normalize)', fontweight='bold')
ax.set_xlabel('Özellik')
ax.set_ylabel('Drift Metriği')

# Sağ: Radar/Spider chart yerine polar bar chart
ax = axes[1]
ozellikler = sayisal_sutunlar
ks_norm    = ks_vals.values
psi_norm   = (psi_vals.values / 2).clip(0, 1)
jsd_norm   = jsd_vals.values

x = np.arange(len(ozellikler))
genislik = 0.25

ax.bar(x - genislik,   ks_norm,  genislik, label='KS',  color='#3498db', alpha=0.8)
ax.bar(x,              psi_norm, genislik, label='PSI/2',color='#e67e22', alpha=0.8)
ax.bar(x + genislik,   jsd_norm, genislik, label='JSD', color='#9b59b6', alpha=0.8)

ax.axhline(y=0.1, color='orange', linestyle='--', lw=1.5, alpha=0.7, label='Orta eşik')
ax.axhline(y=0.2, color='red',    linestyle='--', lw=1.5, alpha=0.7, label='Kritik eşik')

ax.set_title('Drift Metriklerinin Karşılaştırması\n(Tüm Özellikler)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(ozellikler, rotation=15)
ax.set_ylabel('Normalize Drift Skoru')
ax.legend(fontsize=9)
ax.set_ylim(0, max(ks_norm.max(), psi_norm.max(), jsd_norm.max()) * 1.3)

plt.suptitle('Drift Özet Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🤖 Bölüm 5: Makine Öğrenmesi Tabanlı Drift Tespiti (Domain Classifier)

> **Fikir:** Eğer bir sınıflandırıcı, bir verinin eğitim mi yoksa production mı olduğunu yüksek doğrulukla tahmin edebiliyorsa, bu iki veri seti arasında belirgin fark var demektir → **Drift var!**

```
Referans veri → Etiket: 0
Production veri → Etiket: 1
        ↓
İkili sınıflandırma modeli eğit
        ↓
AUC > 0.7 → Drift var!
AUC ≈ 0.5 → Drift yok (rastgele tahmin ediyor)
```

In [ ]:
# =============================================================================
# BÖLÜM 5: Domain Classifier Yöntemi
# "Hangi veri setinden geldiğini tahmin edebilir miyim?"
# =============================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.model_selection import cross_val_score

# Sayısal özellikleri seç ve birleştir
ref_X  = referans_df[sayisal_sutunlar].copy()
prod_X = production_df[sayisal_sutunlar].copy()

# Etiketler: Referans=0, Production=1
ref_X['domain']  = 0  # Eğitim verisi
prod_X['domain'] = 1  # Production verisi

# Birleştir ve karıştır
combined = pd.concat([ref_X, prod_X], ignore_index=True).sample(frac=1, random_state=42)
X_domain = combined[sayisal_sutunlar]
y_domain = combined['domain']

# Eğitim/Test ayrımı
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_domain, y_domain, test_size=0.3, random_state=42, stratify=y_domain
)

# Domain sınıflandırıcı eğit (Random Forest)
domain_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
domain_clf.fit(X_train_d, y_train_d)

# Tahminler ve değerlendirme
y_pred_proba = domain_clf.predict_proba(X_test_d)[:, 1]
y_pred       = domain_clf.predict(X_test_d)

auc_score = roc_auc_score(y_test_d, y_pred_proba)
acc_score = accuracy_score(y_test_d, y_pred)

print("🤖 DOMAIN CLASSIFIER DRIFT TESPİTİ")
print("="*50)
print(f"Domain Sınıflandırıcı Doğruluğu: {acc_score:.4f} ({acc_score*100:.1f}%)")
print(f"ROC-AUC Skoru:                    {auc_score:.4f}")
print()

if auc_score > 0.7:
    print("🔴 SONUÇ: Güçlü Drift Tespit Edildi!")
    print(f"   Model, verilerin eğitim mi production mu olduğunu")
    print(f"   {auc_score*100:.1f}% AUC ile ayırt edebiliyor → Dağılımlar belirgin şekilde farklı!")
elif auc_score > 0.6:
    print("🟡 SONUÇ: Orta Düzey Drift Tespit Edildi")
    print(f"   AUC = {auc_score:.3f} — İzleme ve inceleme önerilir")
else:
    print("🟢 SONUÇ: Drift Tespit Edilmedi")
    print(f"   AUC ≈ 0.5 → Model rastgele tahmin yapıyor, dağılımlar benzer")

In [ ]:
# =============================================================================
# Domain Classifier — ROC Eğrisi ve Özellik Önem Grafiği
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: ROC Eğrisi
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test_d, y_pred_proba)
ax.plot(fpr, tpr, color='darkorchid', lw=2.5, label=f'ROC Eğrisi (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Rastgele tahmin (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='darkorchid')

# Drift eşik çizgisi
ax.axhline(y=0.7, color='orange', linestyle=':', lw=1.5, alpha=0.7)
ax.axvline(x=0.3, color='orange', linestyle=':', lw=1.5, alpha=0.7)

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'Domain Classifier — ROC Eğrisi\n(AUC = {auc_score:.3f} → {"Drift VAR 🔴" if auc_score > 0.7 else "Drift Yok 🟢"})',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Sağ: Özellik Önem Grafiği
ax = axes[1]
importances = pd.Series(domain_clf.feature_importances_, index=sayisal_sutunlar)
importances.sort_values(ascending=True).plot(kind='barh', ax=ax, 
                                              color=['#e74c3c' if v > importances.mean() else '#3498db' 
                                                     for v in importances.sort_values(ascending=True).values],
                                              alpha=0.85)

ax.axvline(x=importances.mean(), color='gray', linestyle='--', lw=1.5, 
           label=f'Ortalama: {importances.mean():.3f}')
ax.set_title('Hangi Özellik En Çok Drift Yaşadı?\n(Domain Classifier Özellik Önemi)', fontweight='bold')
ax.set_xlabel('Önem Skoru')
ax.legend()

# En önemli özelliği vurgula
en_onemli = importances.idxmax()
print(f"\n🎯 En fazla drift yaşayan özellik: '{en_onemli}' (Önem: {importances.max():.3f})")

plt.tight_layout()
plt.show()

---
## ⚙️ Bölüm 6: Evidently AI ile Otomatik Drift Raporu

> **Evidently AI**, ML modeli izleme ve drift tespiti için açık kaynaklı bir kütüphanedir.
> Tek satır kodla kapsamlı HTML raporları üretir.

In [ ]:
# =============================================================================
# BÖLÜM 6: Evidently AI ile Otomatik Drift Raporu
# =============================================================================

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, DataQualityPreset
    from evidently.metrics import DataDriftTable, ColumnDriftMetric
    
    EVIDENTLY_MEVCUT = True
    print("✅ Evidently AI başarıyla yüklendi!")
except ImportError:
    EVIDENTLY_MEVCUT = False
    print("⚠️  Evidently kurulu değil. Kurmak için: pip install evidently")
    print("    Bu bölüm atlanacak, diğer bölümler çalışmaya devam ediyor.")

if EVIDENTLY_MEVCUT:
    # Sadece sayısal sütunlarla rapor oluştur
    ref_evidently  = referans_df[sayisal_sutunlar].copy()
    prod_evidently = production_df[sayisal_sutunlar].copy()
    
    # Kapsamlı Drift Raporu
    drift_raporu = Report(metrics=[
        DataDriftPreset(),          # Tüm özelliklerin drift analizi
    ])
    
    drift_raporu.run(reference_data=ref_evidently, current_data=prod_evidently)
    
    # HTML olarak kaydet
    drift_raporu.save_html('evidently_drift_raporu.html')
    print("✅ Evidently Drift Raporu 'evidently_drift_raporu.html' olarak kaydedildi!")
    print("   Tarayıcınızda açarak interaktif görselleştirmeyi inceleyebilirsiniz.")
    
    # JSON formatında da sonuçları al
    rapor_json = drift_raporu.as_dict()
    print()
    print("📊 Evidently Drift Özeti:")
    print("-"*40)
    
    # Drift sonuçlarını yazdır
    for metric_result in rapor_json.get('metrics', []):
        if 'DatasetDriftMetric' in str(metric_result.get('metric', '')):
            result = metric_result.get('result', {})
            drift_count = result.get('number_of_drifted_columns', 'N/A')
            total_cols  = result.get('number_of_columns', 'N/A')
            dataset_drift = result.get('dataset_drift', False)
            print(f"  Drift Yaşayan Sütun Sayısı: {drift_count}/{total_cols}")
            print(f"  Genel Dataset Drift: {'EVET ⚠️' if dataset_drift else 'HAYIR ✅'}")
else:
    print()
    print("📌 Evidently kurulmadan devam ediliyor...")
    print("   Diğer yöntemlerimiz (KS, PSI, Chi2, Domain Classifier) zaten aktif.")

---
## 🏭 Bölüm 7: Gerçek Dünya Senaryosu — End-to-End Pipeline

Bu bölümde eksiksiz bir drift izleme sistemi kuruyoruz:
1. Model eğit
2. Production'da çalıştır
3. Performans düşüşünü gözlemle
4. Drift'i tespit et
5. Uyarı üret

In [ ]:
# =============================================================================
# BÖLÜM 7: End-to-End Drift İzleme Pipeline'ı
# =============================================================================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

print("=" * 60)
print("🏭 END-TO-END DRIFT İZLEME SİSTEMİ")
print("=" * 60)

# ---- ADIM 1: Referans veriyle model eğit ----
print("\n📌 ADIM 1: Model Eğitimi (Referans Veri)")
print("-" * 40)

X_ref = referans_df[sayisal_sutunlar]
y_ref = referans_df['temerrut']

X_train, X_val, y_train, y_val = train_test_split(
    X_ref, y_ref, test_size=0.2, random_state=42, stratify=y_ref
)

# Pipeline: Ölçekleme + Model
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
val_f1  = f1_score(y_val, model.predict(X_val), average='weighted')
val_acc = accuracy_score(y_val, model.predict(X_val))

print(f"  ✅ Model eğitimi tamamlandı")
print(f"  Validasyon F1-Score:    {val_f1:.4f}")
print(f"  Validasyon Accuracy:    {val_acc:.4f} ({val_acc*100:.1f}%)")

# ---- ADIM 2: Production'da çalıştır ----
print("\n📌 ADIM 2: Production Tahminleri")
print("-" * 40)

X_prod = production_df[sayisal_sutunlar]
y_prod = production_df['temerrut']

prod_tahminler = model.predict(X_prod)
prod_f1  = f1_score(y_prod, prod_tahminler, average='weighted')
prod_acc = accuracy_score(y_prod, prod_tahminler)

print(f"  Production F1-Score:    {prod_f1:.4f}")
print(f"  Production Accuracy:    {prod_acc:.4f} ({prod_acc*100:.1f}%)")

# ---- ADIM 3: Performans düşüşünü hesapla ----
print("\n📌 ADIM 3: Performans Degradasyonu")
print("-" * 40)

f1_dusus  = val_f1  - prod_f1
acc_dusus = val_acc - prod_acc

print(f"  F1 Düşüşü:      {f1_dusus:+.4f} ({f1_dusus/val_f1*100:+.1f}%)")
print(f"  Accuracy Düşüşü: {acc_dusus:+.4f} ({acc_dusus/val_acc*100:+.1f}%)")

if abs(f1_dusus) > 0.05:
    print("  🚨 UYARI: Performans eşiği aşıldı! Drift araştırılmalı!")
elif abs(f1_dusus) > 0.02:
    print("  ⚠️  UYARI: Orta düzey performans düşüşü. İzleme önerilir.")
else:
    print("  ✅ Performans kabul edilebilir sınırlar içinde.")

In [ ]:
# =============================================================================
# Zaman Serisi Boyunca Drift İzleme Simülasyonu
# =============================================================================

print("📌 ADIM 4: Zaman Serisi Drift İzleme (Aylık)")
print("-" * 50)
print("Ekonomik kriz yavaş yavaş etkili oluyor...")
print()

# 12 aylık simülasyon: 0=tamamen normal, 1=tamamen kriz
aylar = [f'Ay {i}' for i in range(1, 13)]
drift_katsayilari = np.linspace(0, 1, 12)  # Kademeli drift

aylik_sonuclar = []

for ay, katsayi in zip(aylar, drift_katsayilari):
    # Kademeli olarak production verisine geç
    n = 300
    normal_n = int(n * (1 - katsayi))
    kriz_n   = n - normal_n
    
    normal_veri = uret_kredi_verisi(n=max(1, normal_n), kriz_modu=False, seed=int(katsayi*100))
    kriz_veri   = uret_kredi_verisi(n=max(1, kriz_n),   kriz_modu=True,  seed=int(katsayi*100+1))
    
    ay_verisi = pd.concat([normal_veri, kriz_veri], ignore_index=True)
    
    # Model performansı
    X_ay = ay_verisi[sayisal_sutunlar]
    y_ay = ay_verisi['temerrut']
    tahminler = model.predict(X_ay)
    ay_acc  = accuracy_score(y_ay, tahminler)
    
    # KS drift skoru (gelir için)
    ks_stat, _ = ks_2samp(referans_df['gelir'].values, ay_verisi['gelir'].values)
    
    # PSI
    psi_val, _ = psi_hesapla(referans_df['gelir'].values, ay_verisi['gelir'].values)
    
    aylik_sonuclar.append({
        'Ay': ay,
        'Model Accuracy': round(ay_acc, 4),
        'KS (Gelir)': round(ks_stat, 4),
        'PSI (Gelir)': round(psi_val, 4),
        'Drift Şiddeti': round(katsayi, 2)
    })

aylik_df = pd.DataFrame(aylik_sonuclar)
print(aylik_df.to_string(index=False))

In [ ]:
# =============================================================================
# Zaman Serisi Drift Görselleştirmesi
# =============================================================================

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

x_pos = np.arange(len(aylar))

# Grafik 1: Model Accuracy
ax = axes[0]
renkler = ['#e74c3c' if v < val_acc - 0.05 else '#f39c12' if v < val_acc - 0.02 else '#2ecc71'
           for v in aylik_df['Model Accuracy']]
bars = ax.bar(x_pos, aylik_df['Model Accuracy'], color=renkler, alpha=0.85)
ax.axhline(y=val_acc,          color='blue',   linestyle='--', lw=2, label=f'Baseline ({val_acc:.3f})')
ax.axhline(y=val_acc - 0.02,   color='orange', linestyle=':', lw=1.5, label='Uyarı Eşiği (-2%)')
ax.axhline(y=val_acc - 0.05,   color='red',    linestyle=':', lw=1.5, label='Kritik Eşik (-5%)')
ax.set_ylabel('Accuracy')
ax.set_title('Model Performansı (Aylık)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.0)

# Grafik 2: KS İstatistiği
ax = axes[1]
ax.plot(x_pos, aylik_df['KS (Gelir)'], 'o-', color='darkorchid', lw=2.5, markersize=8)
ax.fill_between(x_pos, aylik_df['KS (Gelir)'], alpha=0.2, color='darkorchid')
ax.axhline(y=0.1, color='orange', linestyle='--', lw=1.5, label='Uyarı (KS=0.1)')
ax.axhline(y=0.2, color='red',    linestyle='--', lw=1.5, label='Kritik (KS=0.2)')
ax.set_ylabel('KS İstatistiği')
ax.set_title('Gelir Değişkeni KS Drift (Aylık)', fontweight='bold')
ax.legend(fontsize=9)

# Grafik 3: PSI
ax = axes[2]
psi_renkler = ['#e74c3c' if v >= 0.2 else '#f39c12' if v >= 0.1 else '#2ecc71' 
               for v in aylik_df['PSI (Gelir)']]
ax.bar(x_pos, aylik_df['PSI (Gelir)'], color=psi_renkler, alpha=0.85)
ax.axhline(y=0.1, color='orange', linestyle='--', lw=1.5, label='Uyarı (PSI=0.1)')
ax.axhline(y=0.2, color='red',    linestyle='--', lw=1.5, label='Kritik (PSI=0.2)')
ax.set_ylabel('PSI Değeri')
ax.set_xlabel('Ay')
ax.set_title('Gelir Değişkeni PSI (Aylık)', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(aylar, rotation=30)
ax.legend(fontsize=9)

# Kriz başlangıcını tüm grafiklere ekle
for ax in axes:
    ax.axvspan(5.5, 11.5, alpha=0.08, color='red')
    ax.text(8.5, ax.get_ylim()[0] + (ax.get_ylim()[1]-ax.get_ylim()[0])*0.05, 
            '← Kriz Dönemi →', ha='center', color='red', fontsize=9, alpha=0.7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Zaman Serisi Drift İzleme\n(12 Aylık Ekonomik Kriz Senaryosu)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🚨 Bölüm 8: Otomatik Drift Uyarı Sistemi

In [ ]:
# =============================================================================
# BÖLÜM 8: Otomatik Drift İzleme ve Uyarı Sistemi
# Gerçek production sistemlerine entegre edilebilecek fonksiyonlar
# =============================================================================

class DriftIzleme:
    """
    Kapsamlı Veri Kayması İzleme Sistemi
    
    Bu sınıf, production ortamında çalışan ML modellerinin
    veri drift'ini otomatik olarak izler ve uyarı üretir.
    
    Kullanım:
    ---------
    izleyici = DriftIzleme(referans_df, sayisal_sutunlar, kategorik_sutunlar)
    rapor = izleyici.drift_raporu_olustur(production_df)
    izleyici.ozet_yazdir(rapor)
    """
    
    def __init__(self, referans_df, sayisal_sutunlar, kategorik_sutunlar=None):
        """
        Parametreler:
        -----------
        referans_df         : Eğitim veri seti (baseline)
        sayisal_sutunlar    : Sürekli değişken isimleri
        kategorik_sutunlar  : Kategorik değişken isimleri (opsiyonel)
        """
        self.referans        = referans_df
        self.sayisal         = sayisal_sutunlar
        self.kategorik       = kategorik_sutunlar or []
        
        # Uyarı eşikleri
        self.esikler = {
            'ks_uyari':   0.10,   # KS < 0.10: tamam, 0.10-0.20: uyarı, > 0.20: kritik
            'ks_kritik':  0.20,
            'psi_uyari':  0.10,   # PSI < 0.10: tamam
            'psi_kritik': 0.20,
            'p_value':    0.05    # p-value eşiği
        }
    
    def _ks_kontrol(self, production_df):
        """Sayısal değişkenler için KS testi yapar."""
        sonuclar = {}
        for sutun in self.sayisal:
            ks_stat, p_val = ks_2samp(
                self.referans[sutun].dropna(),
                production_df[sutun].dropna()
            )
            
            if ks_stat >= self.esikler['ks_kritik']:
                seviye = 'KRİTİK 🔴'
            elif ks_stat >= self.esikler['ks_uyari']:
                seviye = 'UYARI 🟡'
            else:
                seviye = 'TAMAM 🟢'
            
            sonuclar[sutun] = {
                'ks_stat': round(ks_stat, 4),
                'p_value': round(p_val, 6),
                'drift':   p_val < self.esikler['p_value'],
                'seviye':  seviye
            }
        return sonuclar
    
    def _psi_kontrol(self, production_df):
        """Sayısal değişkenler için PSI hesaplar."""
        sonuclar = {}
        for sutun in self.sayisal:
            psi_val, _ = psi_hesapla(
                self.referans[sutun].values,
                production_df[sutun].values
            )
            
            if psi_val >= self.esikler['psi_kritik']:
                seviye = 'KRİTİK 🔴'
            elif psi_val >= self.esikler['psi_uyari']:
                seviye = 'UYARI 🟡'
            else:
                seviye = 'TAMAM 🟢'
            
            sonuclar[sutun] = {
                'psi': round(psi_val, 4),
                'seviye': seviye
            }
        return sonuclar
    
    def _chi2_kontrol(self, production_df):
        """Kategorik değişkenler için Ki-Kare testi yapar."""
        sonuclar = {}
        for sutun in self.kategorik:
            ref_counts  = self.referans[sutun].value_counts().sort_index()
            prod_counts = production_df[sutun].value_counts().sort_index()
            tum_cat = ref_counts.index.union(prod_counts.index)
            ref_counts  = ref_counts.reindex(tum_cat, fill_value=0)
            prod_counts = prod_counts.reindex(tum_cat, fill_value=0)
            contingency = np.array([ref_counts.values, prod_counts.values])
            chi2, p_val, _, _ = chi2_contingency(contingency)
            
            sonuclar[sutun] = {
                'chi2':    round(chi2, 4),
                'p_value': round(p_val, 6),
                'drift':   p_val < self.esikler['p_value'],
                'seviye':  'KRİTİK 🔴' if p_val < 0.01 else 'UYARI 🟡' if p_val < 0.05 else 'TAMAM 🟢'
            }
        return sonuclar
    
    def drift_raporu_olustur(self, production_df):
        """
        Kapsamlı drift raporu oluşturur.
        
        Döndürür:
        --------
        rapor : dict — KS, PSI, Chi2 sonuçlarını ve genel özeti içerir
        """
        ks_sonuclari   = self._ks_kontrol(production_df)
        psi_sonuclari  = self._psi_kontrol(production_df)
        chi2_sonuclari = self._chi2_kontrol(production_df) if self.kategorik else {}
        
        # Genel drift değerlendirmesi
        kritik_sutunlar = [
            s for s, v in ks_sonuclari.items() if 'KRİTİK' in v['seviye']
        ] + [
            s for s, v in psi_sonuclari.items() if 'KRİTİK' in v['seviye']
        ]
        
        uyari_sutunlar = [
            s for s, v in ks_sonuclari.items() if 'UYARI' in v['seviye'] and s not in kritik_sutunlar
        ]
        
        return {
            'ks':    ks_sonuclari,
            'psi':   psi_sonuclari,
            'chi2':  chi2_sonuclari,
            'ozet':  {
                'kritik_sutunlar': list(set(kritik_sutunlar)),
                'uyari_sutunlar':  list(set(uyari_sutunlar)),
                'toplam_drift':    len(set(kritik_sutunlar)) + len(set(uyari_sutunlar)),
                'genel_seviye':    'KRİTİK 🔴' if kritik_sutunlar else 'UYARI 🟡' if uyari_sutunlar else 'TAMAM 🟢'
            }
        }
    
    def ozet_yazdir(self, rapor):
        """Drift raporunu okunabilir formatta yazdırır."""
        print("=" * 65)
        print("🚨 DRIFT İZLEME RAPORU")
        print("=" * 65)
        
        ozet = rapor['ozet']
        print(f"\n🎯 GENEL DURUM: {ozet['genel_seviye']}")
        print(f"   Kritik Sütunlar: {ozet['kritik_sutunlar'] or 'Yok'}")
        print(f"   Uyarı Sütunları: {ozet['uyari_sutunlar'] or 'Yok'}")
        
        print("\n📊 KS TEST SONUÇLARI (Sayısal):")
        print(f"{'Özellik':20s} {'KS':>8s} {'p-değeri':>12s} {'Durum':>15s}")
        print("-" * 60)
        for sutun, v in rapor['ks'].items():
            print(f"  {sutun:18s} {v['ks_stat']:>8.4f} {v['p_value']:>12.6f} {v['seviye']:>15s}")
        
        print("\n📊 PSI SONUÇLARI (Sayısal):")
        print(f"{'Özellik':20s} {'PSI':>8s} {'Durum':>15s}")
        print("-" * 45)
        for sutun, v in rapor['psi'].items():
            print(f"  {sutun:18s} {v['psi']:>8.4f} {v['seviye']:>15s}")
        
        if rapor['chi2']:
            print("\n📊 KI-KARE SONUÇLARI (Kategorik):")
            for sutun, v in rapor['chi2'].items():
                print(f"  {sutun:20s}: Chi²={v['chi2']:.3f}, p={v['p_value']:.4f}  {v['seviye']}")
        
        print("\n💡 ÖNERİLER:")
        if 'KRİTİK' in ozet['genel_seviye']:
            print("  🔴 Model HEMEN yeniden eğitilmeli!")
            print("  🔴 Kritik özellikleri veri toplama ekibiyle paylaşın")
            print("  🔴 A/B test başlatmayı değerlendirin")
        elif 'UYARI' in ozet['genel_seviye']:
            print("  🟡 Yakın vadede yeniden eğitim planlanmalı")
            print("  🟡 İzleme sıklığı artırılmalı")
        else:
            print("  🟢 Model stabil, rutin izleme yeterli")
        print("=" * 65)


# Sistemi kullan
izleyici = DriftIzleme(
    referans_df     = referans_df,
    sayisal_sutunlar = sayisal_sutunlar,
    kategorik_sutunlar = ['egitim_duzeyi', 'ev_sahibi', 'temerrut']
)

rapor = izleyici.drift_raporu_olustur(production_df)
izleyici.ozet_yazdir(rapor)

---
## 🛡️ Bölüm 9: Drift'e Karşı Önlemler ve Stratejiler

In [ ]:
# =============================================================================
# BÖLÜM 9: Yeniden Eğitim ile Drift'in Etkisini Azaltma
# =============================================================================

from sklearn.metrics import f1_score

print("🛡️  DRIFT KARŞI ÖNLEM: MODELİ YENİDEN EĞİT")
print("=" * 55)

# Strateji 1: Sadece eski veriyle eğitilmiş model
f1_eski = f1_score(y_prod, model.predict(X_prod), average='weighted')

# Strateji 2: Sadece yeni production verisiyle yeniden eğit
X_prod_train, X_prod_test, y_prod_train, y_prod_test = train_test_split(
    X_prod, y_prod, test_size=0.3, random_state=42
)
model_yeni = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(n_estimators=100, random_state=42))
])
model_yeni.fit(X_prod_train, y_prod_train)
f1_yeni = f1_score(y_prod_test, model_yeni.predict(X_prod_test), average='weighted')

# Strateji 3: Eski + Yeni veriyle karma eğitim (warm start)
X_karma = pd.concat([X_ref, X_prod_train], ignore_index=True)
y_karma = pd.concat([y_ref, y_prod_train], ignore_index=True)
model_karma = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(n_estimators=100, random_state=42))
])
model_karma.fit(X_karma, y_karma)
f1_karma = f1_score(y_prod_test, model_karma.predict(X_prod_test), average='weighted')

# Sonuçlar
print(f"\n  Eski Model (drift sonrası):           F1 = {f1_eski:.4f}  ← Bozulmuş!")
print(f"  Yeni Veriyle Yeniden Eğitilmiş:       F1 = {f1_yeni:.4f}  ← İyileşti")
print(f"  Karma (Eski+Yeni) Eğitilmiş:          F1 = {f1_karma:.4f}  ← Dengeli")

print(f"\n  📈 Yeniden Eğitim Kazancı:  +{(f1_yeni - f1_eski):.4f} F1 puan")
print(f"  📈 Karma Eğitim Kazancı:    +{(f1_karma - f1_eski):.4f} F1 puan")

# Görselleştir
fig, ax = plt.subplots(figsize=(9, 4))
stratejiler = ['Eski Model\n(Drift Sonrası)', 'Yeni Veriyle\nYeniden Eğitim', 'Karma\n(Eski+Yeni)']
f1_degerleri = [f1_eski, f1_yeni, f1_karma]
renkler = ['#e74c3c', '#2ecc71', '#3498db']

bars = ax.bar(stratejiler, f1_degerleri, color=renkler, alpha=0.85, width=0.5)
for bar, val in zip(bars, f1_degerleri):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('F1 Skoru (Ağırlıklı)')
ax.set_title('Drift Karşı Strateji Karşılaştırması\n(Yeniden Eğitim Faydası)', fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 📋 Bölüm 10: Nihai Özet ve En İyi Pratikler

In [ ]:
# =============================================================================
# BÖLÜM 10: Kapsamlı Sonuç Özeti
# =============================================================================

print("=" * 70)
print("📋 KAPSAMLI DRIFT TESPİT SONUÇLARI ÖZETİ")
print("=" * 70)

print("\n1️⃣  UYGULANAN YÖNTEMLER VE SONUÇLARI")
print("-" * 50)

# KS Testi özeti
kritik_ks = ks_sonuclari[ks_sonuclari['Drift Şiddeti'] == '🔴 Yüksek']['Özellik'].tolist()
print(f"  KS Testi (Kolmogorov-Smirnov):")
print(f"    → Drift Tespit Edilen: {ks_sonuclari[ks_sonuclari['Drift Var mı?'] == 'EVET ⚠️']['Özellik'].tolist()}")
print(f"    → Kritik Seviye:       {kritik_ks or 'Yok'}")

# PSI özeti
kritik_psi = psi_df[psi_df['Durum'].str.contains('Kritik')]['Özellik'].tolist() if 'Kritik' in str(psi_df['Durum'].values) else psi_df[psi_df['PSI Değeri'] >= 0.2]['Özellik'].tolist()
print(f"\n  PSI (Population Stability Index):")
print(f"    → Kritik PSI (≥0.2):  {psi_df[psi_df['PSI Değeri'] >= 0.2]['Özellik'].tolist() or 'Yok'}")
print(f"    → Uyarı PSI (0.1-0.2): {psi_df[(psi_df['PSI Değeri'] >= 0.1) & (psi_df['PSI Değeri'] < 0.2)]['Özellik'].tolist() or 'Yok'}")

# Chi-Square özeti
print(f"\n  Ki-Kare Testi (Kategorik):")
print(f"    → Drift Tespit Edilen: {chi_sonuclari[chi_sonuclari['Drift Var mı?'] == 'EVET ⚠️']['Özellik'].tolist()}")

# Domain Classifier
print(f"\n  Domain Classifier (ML Tabanlı):")
print(f"    → AUC Skoru: {auc_score:.4f} ({'Güçlü Drift' if auc_score > 0.7 else 'Orta Drift' if auc_score > 0.6 else 'Düşük Drift'})")

print("\n2️⃣  YÖNTEM KARŞILAŞTIRMA TABLOSU")
print("-" * 70)
yontem_tablosu = [
    ["KS Testi",          "Sürekli",        "Basit, yorumlanması kolay", "Tek boyutlu"],
    ["Chi-Square",        "Kategorik",      "Kategorik için standart",   "Bin seçimine duyarlı"],
    ["PSI",               "Sürekli",        "Finans standardı",          "Bin sayısı etkiler"],
    ["JS Divergence",     "Her ikisi",      "Simetrik, 0-1 arası",       "Bin seçimi gerekir"],
    ["Domain Classifier", "Her ikisi",      "Çok boyutlu, güçlü",       "Yorumlanması zor"]
]
print(f"{'Yöntem':22s} {'Değişken':12s} {'Avantaj':28s} {'Dezavantaj'}")
print("-" * 80)
for row in yontem_tablosu:
    print(f"  {row[0]:20s} {row[1]:12s} {row[2]:28s} {row[3]}")

print()
print("3️⃣  ÖNERİLEN DRIFT İZLEME STRATEJİSİ")
print("-" * 50)
stratejiler = [
    "1. Günlük/Haftalık PSI ve KS testleri otomatik çalıştır",
    "2. Uyarı eşiği aşıldığında e-posta/Slack bildirimi gönder",
    "3. Model performans metrikleri (F1, AUC) sürekli izle",
    "4. PSI > 0.2 → Hemen yeniden eğitim pipeline'ını başlat",
    "5. MLflow/Weights&Biases ile drift geçmişini kaydet",
    "6. Evidently AI / Alibi Detect / NannyML gibi araçlar kullan",
    "7. Concept drift için model tahmin dağılımını da izle",
    "8. A/B test ile yeni model vs. eski model karşılaştır"
]
for s in stratejiler:
    print(f"  ✅ {s}")

print("\n" + "=" * 70)
print("🎓 DATA DRIFT TESPİTİ NOTEBOOK TAMAMLANDI!")
print("=" * 70)

---

## 📚 Kaynaklar ve İleri Okuma

### 🔧 Araçlar
| Araç | Açıklama | Link |
|------|----------|------|
| **Evidently AI** | Açık kaynak ML izleme | evidently.ai |
| **Alibi Detect** | Drift & anomali tespiti | seldon.io |
| **NannyML** | Performansız drift tespiti | nannyml.com |
| **Deepchecks** | Veri & model validasyonu | deepchecks.com |
| **WhyLogs** | Hafif profilleme | whylabs.ai |

### 📖 Temel Kavramlar
- **Covariate Drift:** P(X) değişir, P(y|X) sabit
- **Label Drift:** P(y) değişir
- **Concept Drift:** P(y|X) değişir
- **PSI Eşiği:** < 0.1 (tamam), 0.1-0.2 (uyarı), > 0.2 (kritik)
- **KS p-value:** < 0.05 → istatistiksel olarak anlamlı drift

### 🏷️ Etiketler
`#MachineLearning` `#MLOps` `#DataDrift` `#ModelMonitoring` `#Python` `#Statistics`

---
*Bu notebook, production ML sistemlerinde veri kayması tespiti için kapsamlı bir başlangıç noktası sağlar. Gerçek uygulamalarda domain-specific eşikler ve özel izleme gereksinimleri göz önünde bulundurulmalıdır.*